# Data Preprocessing: Labels and Metadata

This notebook is the first clean version of the CXRaide thesis preprocessing workflow.

It focuses only on data preprocessing and labels. It does not include model training.

What this notebook does:

1. Set up paths.
2. Optionally download metadata from Kaggle.
3. Load NIH bounding box metadata.
4. Load VinBig annotation metadata.
5. Normalize labels into one CXRaide label format.
6. Convert bounding boxes into one shared format.
7. Combine NIH and VinBig metadata.
8. Save clean metadata files for review and BigQuery.

The goal is to make the preprocessing easy to follow before turning it into a production pipeline later.

## 1. Import Libraries

If needed, install the notebook dependencies first:

```bash
pip install pandas pyarrow kaggle
```

In [ ]:
from pathlib import Path

import pandas as pd

## 2. Set Project Paths

The raw CSV files should be placed here:

- NIH metadata: `data/raw/nih/`
- VinBig metadata: `data/raw/vinbig/`

The clean outputs will be saved here:

- `data/processed/`

In [ ]:
PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
NIH_RAW_DIR = RAW_DIR / "nih"
VINBIG_RAW_DIR = RAW_DIR / "vinbig"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

NIH_RAW_DIR, VINBIG_RAW_DIR, PROCESSED_DIR

## 3. Optional: Download Metadata From Kaggle

Kaggle does not work like a normal cloud image store. The API downloads files into your machine or runtime.

For this first preprocessing notebook, we only need metadata CSV files.

Before running these commands:

- install the Kaggle package;
- add your `kaggle.json` credentials;
- accept the VinBig competition rules on Kaggle if you need `train.csv`.

In [ ]:
# Run these only when your Kaggle credentials are configured.
# You can also manually download the CSV files and place them in data/raw/nih and data/raw/vinbig.

# !kaggle datasets download -d nih-chest-xrays/data -f BBox_List_2017.csv -p data/raw/nih
# !kaggle datasets download -d nih-chest-xrays/data -f Data_Entry_2017.csv -p data/raw/nih
# !kaggle competitions download -c vinbigdata-chest-xray-abnormalities-detection -f train.csv -p data/raw/vinbig

## 4. Label Mapping

NIH and VinBig do not use exactly the same disease labels. This mapping converts them into one shared CXRaide label format.

The first version keeps these labels:

- Atelectasis
- Cardiomegaly
- Consolidation
- Infiltration
- Nodule/Mass
- Pleural thickening
- Pneumothorax
- Pulmonary fibrosis
- Pleural effusion

In [ ]:
CANONICAL_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Infiltration",
    "Nodule/Mass",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
    "Pleural effusion",
]

LABEL_MAPPING = {
    "Mass": "Nodule/Mass",
    "Nodule": "Nodule/Mass",
    "Nodule Mass": "Nodule/Mass",
    "Effusion": "Pleural effusion",
    "Fibrosis": "Pulmonary fibrosis",
    "Pleural_Thickening": "Pleural thickening",
    "Pleural thickening": "Pleural thickening",
}

def normalize_label(label):
    label = str(label).strip()
    return LABEL_MAPPING.get(label, label)

CANONICAL_LABELS

## 5. Load NIH Bounding Box Metadata

NIH bounding boxes usually come from `BBox_List_2017.csv`.

Important NIH columns:

- `Image Index`: image filename
- `Finding Label`: disease label
- `Bbox [x`: left coordinate
- `y`: top coordinate
- `w`: box width
- `h]`: box height

NIH stores boxes as `x, y, width, height`. We will convert them to `x_min, y_min, x_max, y_max`.

In [ ]:
nih_bbox_path = NIH_RAW_DIR / "BBox_List_2017.csv"

if not nih_bbox_path.exists():
    nih_bbox_path = NIH_RAW_DIR / "BBox_list_2017.csv"

nih_bbox_path

In [ ]:
nih_raw = pd.read_csv(nih_bbox_path)
nih_raw.head()

## 6. Clean NIH Labels and Boxes

In [ ]:
nih = nih_raw.copy()

nih = nih.rename(columns={
    "Image Index": "image_id",
    "Finding Label": "class_name",
    "Bbox [x": "x_min",
    "y": "y_min",
    "w": "box_width",
    "h]": "box_height",
})

nih["image_id"] = nih["image_id"].astype(str).str.replace(".png", "", regex=False)
nih["class_name"] = nih["class_name"].apply(normalize_label)

nih = nih[nih["class_name"].isin(CANONICAL_LABELS)].copy()

# NIH images are commonly 1024 x 1024. The thesis preprocessing resized to 512 x 512 first.
ORIGINAL_IMAGE_SIZE = 1024
TARGET_IMAGE_SIZE = 512
scale = TARGET_IMAGE_SIZE / ORIGINAL_IMAGE_SIZE

nih["x_min"] = nih["x_min"].astype(float) * scale
nih["y_min"] = nih["y_min"].astype(float) * scale
nih["x_max"] = nih["x_min"] + nih["box_width"].astype(float) * scale
nih["y_max"] = nih["y_min"] + nih["box_height"].astype(float) * scale

nih["source_dataset"] = "NIH"
nih["width"] = TARGET_IMAGE_SIZE
nih["height"] = TARGET_IMAGE_SIZE

nih_clean = nih[[
    "source_dataset",
    "image_id",
    "class_name",
    "x_min",
    "y_min",
    "x_max",
    "y_max",
    "width",
    "height",
]].copy()

nih_clean.head()

In [ ]:
nih_clean["class_name"].value_counts()

## 7. Load VinBig Annotation Metadata

VinBig annotations usually come from `train.csv`.

Important VinBig columns:

- `image_id`
- `class_name`
- `x_min`
- `y_min`
- `x_max`
- `y_max`

VinBig already uses `x_min, y_min, x_max, y_max`, so we mainly normalize labels and remove labels we are not using yet.

In [ ]:
vinbig_path = VINBIG_RAW_DIR / "train.csv"
vinbig_path

In [ ]:
vinbig_raw = pd.read_csv(vinbig_path)
vinbig_raw.head()

## 8. Clean VinBig Labels and Boxes

Rows labeled `No finding` are not used in this first bounding-box label dataset because they do not provide disease boxes.

In [ ]:
vinbig = vinbig_raw.copy()

vinbig["image_id"] = vinbig["image_id"].astype(str).str.replace(".png", "", regex=False)
vinbig["class_name"] = vinbig["class_name"].apply(normalize_label)
vinbig = vinbig[vinbig["class_name"].isin(CANONICAL_LABELS)].copy()

for column in ["x_min", "y_min", "x_max", "y_max"]:
    vinbig[column] = vinbig[column].astype(float)

# If width and height exist, rescale to 512 x 512. Otherwise, keep the coordinates as provided.
if {"width", "height"}.issubset(vinbig.columns):
    x_scale = TARGET_IMAGE_SIZE / vinbig["width"].astype(float)
    y_scale = TARGET_IMAGE_SIZE / vinbig["height"].astype(float)
    vinbig["x_min"] = vinbig["x_min"] * x_scale
    vinbig["x_max"] = vinbig["x_max"] * x_scale
    vinbig["y_min"] = vinbig["y_min"] * y_scale
    vinbig["y_max"] = vinbig["y_max"] * y_scale

vinbig["source_dataset"] = "VinBig"
vinbig["width"] = TARGET_IMAGE_SIZE
vinbig["height"] = TARGET_IMAGE_SIZE

vinbig_clean = vinbig[[
    "source_dataset",
    "image_id",
    "class_name",
    "x_min",
    "y_min",
    "x_max",
    "y_max",
    "width",
    "height",
]].copy()

vinbig_clean.head()

In [ ]:
vinbig_clean["class_name"].value_counts()

## 9. Combine NIH and VinBig Metadata

Both datasets now use the same columns and the same label names, so they can be combined.

In [ ]:
combined_annotations = pd.concat([nih_clean, vinbig_clean], ignore_index=True)

combined_annotations = combined_annotations.drop_duplicates(
    subset=["source_dataset", "image_id", "class_name", "x_min", "y_min", "x_max", "y_max"]
).reset_index(drop=True)

combined_annotations.head()

In [ ]:
combined_annotations.shape

In [ ]:
combined_annotations.groupby(["source_dataset", "class_name"]).size().unstack(fill_value=0)

## 10. Create a Simple Image-Level Label Table

This table has one row per image and one column per label.

This is useful for checking which diseases appear in each image and is easier to understand than the bounding-box table.

In [ ]:
image_labels = (
    combined_annotations
    .assign(value=1)
    .pivot_table(
        index=["source_dataset", "image_id"],
        columns="class_name",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

image_labels.head()

## 11. Save Clean Outputs

This saves both CSV and Parquet.

Use CSV when you want to inspect files manually.

Use Parquet when uploading to BigQuery because it preserves numeric types better.

In [ ]:
annotations_csv_path = PROCESSED_DIR / "clean_annotations.csv"
annotations_parquet_path = PROCESSED_DIR / "clean_annotations.parquet"

image_labels_csv_path = PROCESSED_DIR / "image_labels.csv"
image_labels_parquet_path = PROCESSED_DIR / "image_labels.parquet"

combined_annotations.to_csv(annotations_csv_path, index=False)
combined_annotations.to_parquet(annotations_parquet_path, index=False)

image_labels.to_csv(image_labels_csv_path, index=False)
image_labels.to_parquet(image_labels_parquet_path, index=False)

annotations_csv_path, annotations_parquet_path, image_labels_csv_path, image_labels_parquet_path

## 12. BigQuery Note

For BigQuery, I recommend uploading the Parquet files instead of making CSV the main format.

Recommended metadata tables:

1. `clean_annotations`: one row per bounding box.
2. `image_labels`: one row per image with binary label columns.

Later, when you upload images to Google Cloud Storage, add an image URI column like this:

`gs://your-bucket/cxraide/images/{source_dataset}/{image_id}.png`

BigQuery should store metadata and image locations. It should not store the actual image bytes.